# Open-Set Cattle Muzzle Recognition using ResNet-50 (CLAHE Enhanced)

This notebook implements an end-to-end deep learning pipeline for Open-Set Recognition (OSR) on the Cattle Muzzle print dataset. It uses a ResNet-50 model trained with **Contrast Limited Adaptive Histogram Equalization (CLAHE)** preprocessing.

## Dataset Split Details
- **Total Classes:** 268 cows (classes).
- **Known Classes:** 187 cows for training and validation.
  - **Train/Val Split:** 80% of images for training, 20% for validation.
  - **Data Augmentation:** Applied on the training split of the 187 known classes.
- **Known Test Classes:** 54 cows (subset of the 187 known cows, evaluated using their validation images).
- **Unknown Test Classes:** 27 cows (disjoint set of unseen cows, evaluated using all of their images).

## OSR Classification Strategy
- We train a closed-set classifier with **187 outputs** corresponding to the known training classes.
- During inference, we apply a confidence threshold $\tau=0.5$ on the maximum softmax probability. If the maximum probability is below $\tau$, we reject the sample as **"unknown"** (class index 187).

## Instructions for Kaggle
1. Upload this notebook to your Kaggle account.
2. Add the **BeefCattle_Muzzle_Individualized** dataset to your Kaggle session.
3. Turn on the **GPU Accelerator** (NVIDIA T4 or P100) in Kaggle settings.
4. Run all cells. All plots and the `classification_report_clahe.csv` will be saved in `/kaggle/working`.

In [ ]:
import os
import cv2
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
import timm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

## 1. Setting Up Reproducibility & Device Configuration

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if torch.backends.mps.is_available():
        torch.manual_seed(seed)

set_seed(42)

# Choose device: GPU (CUDA), Apple Silicon (MPS), or CPU
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Dataset Path Configuration

We detect whether we are running on Kaggle or locally, setting paths accordingly. Prints `/kaggle/input` contents and raises an error if no datasets are attached.

In [ ]:
kaggle_path = "/kaggle/input/beefcattle-muzzle-individualized"
local_path = "Cattle Muzzle - DB"

if os.path.exists(kaggle_path):
    DATA_DIR = kaggle_path
    print(f"Running on Kaggle. Using direct dataset path: {DATA_DIR}")
elif os.path.exists("/kaggle/input"):
    input_contents = os.listdir("/kaggle/input")
    print(f"Running on Kaggle. Contents of '/kaggle/input': {input_contents}")
    possible_dirs = [os.path.join("/kaggle/input", d) for d in input_contents if os.path.isdir(os.path.join("/kaggle/input", d))]
    if possible_dirs:
        DATA_DIR = possible_dirs[0]
        print(f"Automatically detected dataset directory: {DATA_DIR}")
    else:
        raise FileNotFoundError(
            "Running on Kaggle, but '/kaggle/input' is empty. Please attach the dataset to your Kaggle notebook session using the '+ Add Data' button on the right panel."
        )
else:
    if os.path.exists("BeefCattle_Muzzle_Individualized"):
        DATA_DIR = "BeefCattle_Muzzle_Individualized"
    else:
        DATA_DIR = local_path
    print(f"Running locally. Dataset path: {DATA_DIR}")

# Auto-unwrap nested directories if dataset zip was uploaded with an inner wrapper folder
def resolve_dataset_dir(path):
    current = path
    while True:
        items = [i for i in os.listdir(current) if not i.startswith('.')]
        if len(items) == 1 and os.path.isdir(os.path.join(current, items[0])):
            current = os.path.join(current, items[0])
        else:
            break
    return current

DATA_DIR = resolve_dataset_dir(DATA_DIR)
print(f"Resolved Dataset Directory: {DATA_DIR}")


## 3. Dynamic Splitting Logic for OSR

We partition the dataset classes into Known classes (187), Known test classes (54), and Unknown test classes (27) dynamically, allowing the code to scale down on smaller datasets for testing.

In [ ]:
def collect_images_by_class(data_dir):
    class_images = {}
    has_splits = all(os.path.exists(os.path.join(data_dir, s)) for s in ['train', 'val', 'test'])
    
    if has_splits:
        print(f"Detected pre-split directories in '{data_dir}'. Merging image paths...")
        splits = ['train', 'val', 'test']
        class_names = set()
        for s in splits:
            split_dir = os.path.join(data_dir, s)
            for d in os.listdir(split_dir):
                if os.path.isdir(os.path.join(split_dir, d)) and not d.startswith('.'):
                    class_names.add(d)
        class_names = sorted(list(class_names))
        
        for cls in class_names:
            class_images[cls] = []
            for s in splits:
                cls_dir = os.path.join(data_dir, s, cls)
                if os.path.exists(cls_dir):
                    for f in os.listdir(cls_dir):
                        fp = os.path.join(cls_dir, f)
                        if os.path.isfile(fp) and not f.startswith('.'):
                            class_images[cls].append(fp)
    else:
        print(f"Detected flat directories in '{data_dir}'.")
        class_names = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d)) and not d.startswith('.')])
        for cls in class_names:
            cls_dir = os.path.join(data_dir, cls)
            class_images[cls] = []
            for f in os.listdir(cls_dir):
                fp = os.path.join(cls_dir, f)
                if os.path.isfile(fp) and not f.startswith('.'):
                    class_images[cls].append(fp)
                    
    return class_images

class_images = collect_images_by_class(DATA_DIR)
all_classes = sorted(list(class_images.keys()))
num_total_classes = len(all_classes)
print(f"Total classes in database: {num_total_classes}")

if num_total_classes == 268:
    num_known = 187
    num_unknown = 27
    num_known_test = 54
else:
    # Scale splits proportionally for local / smaller datasets
    num_known = max(2, int(187 / 268 * num_total_classes))
    num_unknown = max(1, int(27 / 268 * num_total_classes))
    num_known_test = max(1, int(54 / 268 * num_total_classes))
    if num_known + num_unknown > num_total_classes:
        num_known = num_total_classes - num_unknown
    if num_known_test > num_known:
        num_known_test = num_known

print(f"OSR Configurations:")
print(f"  - Training & Validation Classes: {num_known}")
print(f"  - Known Test Classes (subset of known): {num_known_test}")
print(f"  - Unknown Test Classes (unseen): {num_unknown}")

# Deterministic shuffling of classes
shuffled_classes = all_classes.copy()
random.seed(42)
random.shuffle(shuffled_classes)

known_classes = shuffled_classes[:num_known]
unseen_classes = shuffled_classes[num_known:]

known_test_classes = known_classes[:num_known_test]
unknown_test_classes = unseen_classes[:num_unknown]

class_to_idx = {cls: i for i, cls in enumerate(known_classes)}

train_paths, train_labels = [], []
val_paths, val_labels = [], []
known_test_paths, known_test_labels = [], []
unknown_test_paths, unknown_test_labels = [], []

for cls in known_classes:
    paths = class_images[cls]
    random.seed(42)
    random.shuffle(paths)
    
    split_idx = int(len(paths) * 0.8) # 80:20 split
    c_train = paths[:split_idx]
    c_val = paths[split_idx:]
    
    train_paths.extend(c_train)
    train_labels.extend([class_to_idx[cls]] * len(c_train))
    
    val_paths.extend(c_val)
    val_labels.extend([class_to_idx[cls]] * len(c_val))
    
    if cls in known_test_classes:
        known_test_paths.extend(c_val)
        known_test_labels.extend([class_to_idx[cls]] * len(c_val))
        
for cls in unknown_test_classes:
    paths = class_images[cls]
    unknown_test_paths.extend(paths)
    unknown_test_labels.extend([num_known] * len(paths)) # True OSR label for unseen is index num_known

print(f"Image Splits Summary:")
print(f"  - Training Images (187 classes, augmented): {len(train_paths)}")
print(f"  - Validation Images (187 classes): {len(val_paths)}")
print(f"  - Known Test Images (54 classes): {len(known_test_paths)}")
print(f"  - Unknown Test Images (27 classes): {len(unknown_test_paths)}")

## 4. CLAHE Preprocessing & Dataset Transforms

In [ ]:
class ApplyCLAHE:
    """
    Applies Contrast Limited Adaptive Histogram Equalization (CLAHE) on a PIL Image.
    Isolates the luminance (L) channel in LAB color space to preserve color structure.
    """
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, img):
        img_np = np.array(img)
        clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)
        
        if len(img_np.shape) == 2:  # Grayscale image
            enhanced_np = clahe.apply(img_np)
            return Image.fromarray(enhanced_np)
        else:  # RGB image
            lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
            l, a, b = cv2.split(lab)
            cl = clahe.apply(l)
            limg = cv2.merge((cl, a, b))
            enhanced_np = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)
            return Image.fromarray(enhanced_np)

class CowDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        
    def __getitem__(self, idx):
        path = self.image_paths[idx]
        label = self.labels[idx]
        try:
            img = Image.open(path).convert('RGB')
        except Exception as e:
            img = Image.new('RGB', (224, 224), color=0)
            print(f"Warning: Corrupted image {path}: {e}")
            
        if self.transform:
            img = self.transform(img)
        return img, label
        
    def __len__(self):
        return len(self.image_paths)

# Data Augmentation on training, Standard Resizing/Normalization on Val/Test
train_transform = transforms.Compose([
    ApplyCLAHE(),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    ApplyCLAHE(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = CowDataset(train_paths, train_labels, transform=train_transform)
val_dataset = CowDataset(val_paths, val_labels, transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2 if os.name != 'nt' else 0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2 if os.name != 'nt' else 0, pin_memory=True)

## 5. Model Definition (ResNet-50 with Pretrained ImageNet Weights)

In [ ]:
print("Initializing ResNet-50 model...")
model = timm.create_model('resnet50', pretrained=True, num_classes=num_known)
model = model.to(device)

dummy_input = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    dummy_output = model(dummy_input)
print("Model output shape (batch_size=2, num_classes):", dummy_output.shape)

## 6. Optimizer, Loss & Hyperparameters Configuration

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

# Cosine annealing LR scheduler for optimal convergence
epochs = 30
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}
best_val_acc = 0.0
best_model_path = 'best_resnet50_osr_clahe.pth'

## 7. Model Training Loop

In [ ]:
print("Starting training loop...")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_train_loss = running_loss / len(train_dataset)
    epoch_train_acc = correct / total
    scheduler.step()
    
    # Validation Phase
    model.eval()
    running_val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
    epoch_val_loss = running_val_loss / len(val_dataset)
    epoch_val_acc = val_correct / val_total
    
    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)
    
    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.4f} | Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}")
    
    # Checkpointing best weights
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f"  => Saved new best validation model (Acc: {best_val_acc:.4f})")

print(f"Training complete! Best validation accuracy: {best_val_acc:.4f}")

## 8. Plotting Training and Validation Curves

In [ ]:
plt.figure(figsize=(14, 5))

# Loss curves
plt.subplot(1, 2, 1)
plt.plot(range(1, len(history['train_loss']) + 1), history['train_loss'], label='Train Loss', color='#FF6B6B', lw=2.5)
plt.plot(range(1, len(history['val_loss']) + 1), history['val_loss'], label='Val Loss', color='#4D96FF', lw=2.5)
plt.title('Loss Curves (CLAHE Enhanced)', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()

# Accuracy curves
plt.subplot(1, 2, 2)
plt.plot(range(1, len(history['train_acc']) + 1), history['train_acc'], label='Train Acc', color='#FF6B6B', lw=2.5)
plt.plot(range(1, len(history['val_acc']) + 1), history['val_acc'], label='Val Acc', color='#4D96FF', lw=2.5)
plt.title('Accuracy Curves (CLAHE Enhanced)', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()

plt.tight_layout()
plt.savefig('accuracy_loss_curves_clahe.png', dpi=300)
plt.show()

## 9. Evaluation on OSR Test Set

We load the best saved model weights and run predictions on both the Known Test dataset (54 classes) and the Unknown Test dataset (27 classes). If the maximum Softmax probability is less than the threshold $\tau=0.5$, we classify it as `"unknown"` (index `num_known`).

In [ ]:
model.load_state_dict(torch.load(best_model_path))
model.eval()

known_test_dataset = CowDataset(known_test_paths, known_test_labels, transform=val_test_transform)
unknown_test_dataset = CowDataset(unknown_test_paths, unknown_test_labels, transform=val_test_transform)

known_test_loader = DataLoader(known_test_dataset, batch_size=16, shuffle=False)
unknown_test_loader = DataLoader(unknown_test_dataset, batch_size=16, shuffle=False)

def extract_predictions(loader):
    all_probs = []
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            max_probs, preds = torch.max(probs, dim=1)
            all_probs.extend(max_probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(labels.numpy())
    return np.array(all_probs), np.array(all_preds), np.array(all_targets)

print("Evaluating known test set...")
k_probs, k_preds, k_targets = extract_predictions(known_test_loader)
print("Evaluating unknown test set...")
u_probs, u_preds, u_targets = extract_predictions(unknown_test_loader)

combined_probs = np.concatenate([k_probs, u_probs])
combined_raw_preds = np.concatenate([k_preds, u_preds])
combined_targets = np.concatenate([k_targets, u_targets])

# Set threshold for OSR
osr_threshold = 0.5
combined_final_preds = np.where(combined_probs >= osr_threshold, combined_raw_preds, num_known)

total_accuracy = np.mean(combined_final_preds == combined_targets)
print(f"OSR Combined Test Set Accuracy: {total_accuracy:.4f}")

## 10. Confusion Matrix (OSR Summary & Detailed)

In [ ]:
# 1. OSR Binary Confusion Matrix (Known vs Unknown)
bin_targets = np.where(combined_targets == num_known, 1, 0) # 1 = Unknown, 0 = Known
bin_preds = np.where(combined_final_preds == num_known, 1, 0)
bin_cm = confusion_matrix(bin_targets, bin_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(bin_cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Known', 'Unknown'], yticklabels=['Known', 'Unknown'])
plt.title('Binary OSR Detection (Known vs Unknown)', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig('binary_confusion_matrix_clahe.png', dpi=300)
plt.show()

# 2. Detailed Confusion Matrix (for classes present in known test + unknown)
unique_targets = sorted(list(set(combined_targets)))

# Take a subset of first 10 evaluated classes + unknown to keep heatmap clean
subset_size = min(10, len(unique_targets) - 1)
display_classes = unique_targets[:subset_size] + [num_known]
display_names = [f"cow_{c}" if c != num_known else "unknown" for c in display_classes]

idx_filter = np.isin(combined_targets, display_classes) & np.isin(combined_final_preds, display_classes)
if np.sum(idx_filter) > 0:
    cm_sub = confusion_matrix(combined_targets[idx_filter], combined_final_preds[idx_filter], labels=display_classes)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_sub, annot=True, fmt='d', cmap='Blues', xticklabels=display_names, yticklabels=display_names)
    plt.title('Multi-Class OSR Confusion Matrix Subset', fontsize=12, fontweight='bold', pad=12)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
plt.savefig('multiclass_confusion_matrix_subset_clahe.png', dpi=300)
plt.show()

## 11. Classification Report & Export to CSV

We generate the full classification report containing precision, recall, and f1-score for all 187 known classes + the unknown class. We then save this report to a CSV file.

In [ ]:
target_names = [f"cow_{cls}" for cls in known_classes] + ["unknown"]
report_dict = classification_report(
    combined_targets, 
    combined_final_preds, 
    labels=list(range(num_known + 1)), 
    target_names=target_names, 
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv('classification_report_clahe.csv', index=True)

print("Open-Set Classification Report Preview:")
print(report_df.iloc[np.concatenate([np.arange(5), [-3, -2, -1]])])
print("\nClassification report successfully saved to 'classification_report_clahe.csv'!")